# SatQuery AI — Deterministic Spectral Perception & Threshold Sanity Checks
**SIH 2026 (PS 26167, ISRO/SAC)**

This interactive notebook demonstrates the deterministic perception layer for Sentinel-2 optical imagery:
1. **`compute_indices(bands)`** — Normalized difference indices (NDVI, NDWI, NDBI) with zero-division protection
2. **Visualization** — 2D spatial maps of each spectral index
3. **Threshold Masking** — Vegetation, Water, and Built-up surface extraction

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is on sys.path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from satquery.perception.spectral_indices import compute_indices
from satquery.perception.sar_backscatter import compute_sar_masks
from satquery.perception.cloud_mask import compute_cloud_mask
from satquery.fusion.optical_sar_fusion import fuse

## 1. Load Sample Sentinel-2 Multi-Spectral Bands
Loading Red, Green, NIR, and SWIR bands from `data/demo_samples/single_optical/`.

In [ ]:
sample_path = ROOT_DIR / "data" / "demo_samples" / "single_optical" / "sample_sentinel2_bands.npz"

if sample_path.exists():
    data = np.load(sample_path)
    bands = {
        "red": data["red"],
        "green": data["green"],
        "blue": data["blue"],
        "nir": data["nir"],
        "swir": data["swir"],
    }
    print(f"Loaded bands from {sample_path} (Shape: {bands['red'].shape})")
else:
    print("Generating test bands inline...")
    H, W = 128, 128
    bands = {
        "red": np.full((H, W), 0.15, dtype=np.float32),
        "green": np.full((H, W), 0.15, dtype=np.float32),
        "blue": np.full((H, W), 0.15, dtype=np.float32),
        "nir": np.full((H, W), 0.20, dtype=np.float32),
        "swir": np.full((H, W), 0.18, dtype=np.float32),
    }
    # Water quadrant
    bands["nir"][0:64, 0:64] = 0.02
    bands["green"][0:64, 0:64] = 0.12
    # Forest quadrant
    bands["nir"][64:128, 0:64] = 0.70
    bands["red"][64:128, 0:64] = 0.05
    # Urban quadrant
    bands["swir"][64:128, 64:128] = 0.50
    bands["nir"][64:128, 64:128] = 0.25

## 2. Compute Deterministic Spectral Indices
- **NDVI** = $(NIR - Red) / (NIR + Red)$
- **NDWI** = $(Green - NIR) / (Green + NIR)$
- **NDBI** = $(SWIR - NIR) / (SWIR + NIR)$

In [ ]:
indices = compute_indices(bands)
ndvi = indices["ndvi"]
ndwi = indices["ndwi"]
ndbi = indices["ndbi"]

print(f"NDVI Range: [{ndvi.min():.3f}, {ndvi.max():.3f}] | Mean: {ndvi.mean():.3f}")
print(f"NDWI Range: [{ndwi.min():.3f}, {ndwi.max():.3f}] | Mean: {ndwi.mean():.3f}")
print(f"NDBI Range: [{ndbi.min():.3f}, {ndbi.max():.3f}] | Mean: {ndbi.mean():.3f}")

## 3. Visualize Spectral Index Maps

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Natural color RGB (R, G, B)
rgb = np.stack([bands["red"], bands["green"], bands["blue"]], axis=-1)
rgb_norm = np.clip(rgb / (np.max(rgb) + 1e-6), 0.0, 1.0)
axes[0].imshow(rgb_norm)
axes[0].set_title("True Color (RGB)", fontsize=12, fontweight="bold")
axes[0].axis("off")

# NDVI
im1 = axes[1].imshow(ndvi, cmap="RdYlGn", vmin=-0.5, vmax=0.9)
axes[1].set_title("NDVI (Vegetation Index)", fontsize=12, fontweight="bold")
axes[1].axis("off")
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

# NDWI
im2 = axes[2].imshow(ndwi, cmap="Blues", vmin=-0.8, vmax=0.8)
axes[2].set_title("NDWI (Water Index)", fontsize=12, fontweight="bold")
axes[2].axis("off")
plt.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)

# NDBI
im3 = axes[3].imshow(ndbi, cmap="YlOrRd", vmin=-0.8, vmax=0.6)
axes[3].set_title("NDBI (Built-up Index)", fontsize=12, fontweight="bold")
axes[3].axis("off")
plt.colorbar(im3, ax=axes[3], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

## 4. Threshold Sanity Checks & Coverage Fractions
- **Vegetation**: $NDVI \ge 0.30$
- **Water Body**: $NDWI \ge 0.00$
- **Built-up / Urban**: $NDBI \ge 0.00$

In [ ]:
veg_mask = ndvi >= 0.30
water_mask = ndwi >= 0.00
builtup_mask = ndbi >= 0.00

total_px = float(ndvi.size)
print(f"Vegetation Coverage : {np.sum(veg_mask) / total_px * 100:.2f}%")
print(f"Water Body Coverage : {np.sum(water_mask) / total_px * 100:.2f}%")
print(f"Built-up Coverage   : {np.sum(builtup_mask) / total_px * 100:.2f}%")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(veg_mask, cmap="Greens")
axes[0].set_title("Vegetation Mask (NDVI >= 0.30)", fontweight="bold")
axes[0].axis("off")

axes[1].imshow(water_mask, cmap="Blues")
axes[1].set_title("Water Mask (NDWI >= 0.00)", fontweight="bold")
axes[1].axis("off")

axes[2].imshow(builtup_mask, cmap="Reds")
axes[2].set_title("Built-up Mask (NDBI >= 0.00)", fontweight="bold")
axes[2].axis("off")

plt.tight_layout()
plt.show()